# 05 — Watchlist Predictions

Predicts a rating for every unseen film on the watchlist, for the Streamlit app.

**Input:** the Letterboxd export (`watchlist.csv`), plus `data/interim/viewings.csv` and
`films_enriched.csv` for the rated history
**Output:** watchlist predictions, one row per film

The watchlist goes through the same pipeline as the rated films: matched with the same
rules (`src/tmdb.py`), given the same features, and scored by Model 4 RF refitted on all
1,192 rated viewings. Watchlist API responses are cached in **separate files**, so the
rated-film caches from `02` are never touched.

## 1. Load the watchlist

In [20]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

import numpy as np
from src.features import genre_vocabulary, genre_history

from src.letterboxd import load_export, film_key
from src.tmdb import make_headers, match_all, AUTO_ACCEPT, search_film, fetch_all_details
from src.features import (genre_vocabulary, genre_history, top_n_vocabulary,
                          build_features_F, build_features_H, keyword_scores,
                          add_keyword_score, PLAIN_NUMERIC, LOG_NUMERIC, HISTORY_KEYS)

from sklearn.ensemble import RandomForestRegressor

load_dotenv()
export = load_export(os.getenv("LETTERBOXD_EXPORT_DIR"))

watchlist = export["watchlist"].copy()
watchlist["film_key"] = film_key(watchlist)

rated   = set(film_key(export["ratings"]))
watched = set(film_key(export["watched"]))

print(f"watchlist rows   : {len(watchlist)}")
print(f"unique film_keys : {watchlist['film_key'].nunique()}")
print(f"missing year     : {watchlist['Year'].isna().sum()}")
print(f"already rated    : {watchlist['film_key'].isin(rated).sum()}")
print(f"already watched  : {watchlist['film_key'].isin(watched).sum()}")

watchlist rows   : 4031
unique film_keys : 4022
missing year     : 8
already rated    : 0
already watched  : 0


In [2]:
dupes = watchlist[watchlist["film_key"].duplicated(keep=False)].sort_values("film_key")
print(f"{len(dupes)} rows share a film_key ({dupes['film_key'].nunique()} keys)\n")
print(dupes[["Name", "Year", "Letterboxd URI"]].to_string(index=False))

print("\nmissing year:\n")
print(watchlist[watchlist["Year"].isna()][["Name", "Letterboxd URI"]].to_string(index=False))

10 rows share a film_key (1 keys)

                    Name   Year        Letterboxd URI
              The Castle 1997.0  https://boxd.it/1sWI
              The Castle 1997.0  https://boxd.it/1Pho
                 Polaris    NaN  https://boxd.it/vMS2
       The Memory Police    NaN  https://boxd.it/scb6
         The Governesses    NaN  https://boxd.it/AdVq
           Tower Stories    NaN  https://boxd.it/nzTg
              Love Child    NaN  https://boxd.it/fAP4
The Bookie & the Bruiser    NaN  https://boxd.it/N6Oe
    Here Comes the Flood    NaN  https://boxd.it/qdjm
              Lily May B    NaN https://boxd.it/13ssm

missing year:

                    Name        Letterboxd URI
                 Polaris  https://boxd.it/vMS2
       The Memory Police  https://boxd.it/scb6
         The Governesses  https://boxd.it/AdVq
           Tower Stories  https://boxd.it/nzTg
              Love Child  https://boxd.it/fAP4
The Bookie & the Bruiser  https://boxd.it/N6Oe
    Here Comes the Flood  

In [3]:
undated  = watchlist["film_key"].isna()
collides = watchlist["film_key"].duplicated(keep=False) & ~undated

collisions = watchlist[collides].copy()

films_wl = (watchlist[~undated & ~collides]
            .rename(columns={"Name": "film_title", "Year": "film_year",
                             "Letterboxd URI": "film_uri"})
            [["film_key", "film_title", "film_year", "film_uri"]]
            .reset_index(drop=True))

print(f"excluded, no year      : {undated.sum()}")
print(f"held back, collision   : {collides.sum()} rows, {collisions['film_key'].nunique()} key(s)")
print(f"to match automatically : {len(films_wl)}")

excluded, no year      : 8
held back, collision   : 2 rows, 1 key(s)
to match automatically : 4021


In [4]:
HEADERS = make_headers(os.getenv("TMDB_TOKEN"))

WL_SEARCH_CACHE = Path("data/cache/watchlist_search_raw.json")

wl_matches = match_all(films_wl, headers=HEADERS, cache_path=WL_SEARCH_CACHE)

done — 0 new API calls, 4021 from cache


In [5]:
print(f"films: {len(wl_matches)}\n")
print("confidence breakdown")
for tier, n in wl_matches["confidence"].value_counts().items():
    print(f"  {tier:12s} {n:5d}  ({n/len(wl_matches)*100:5.1f}%)")

wl_review = wl_matches[~wl_matches["confidence"].isin(AUTO_ACCEPT)]
print(f"\nauto-accepted  : {len(wl_matches) - len(wl_review)} "
      f"({(len(wl_matches) - len(wl_review)) / len(wl_matches) * 100:.1f}%)")
print(f"needs review   : {len(wl_review)}")
print(f"no match at all: {wl_matches['tmdb_id'].isna().sum()}")

films: 4021

confidence breakdown
  exact         3673  ( 91.3%)
  year_off       295  (  7.3%)
  weak            49  (  1.2%)
  close            3  (  0.1%)
  no_match         1  (  0.0%)

auto-accepted  : 3968 (98.7%)
needs review   : 53
no match at all: 1


In [6]:
print(wl_review.sort_values(["confidence", "similarity"])[
    ["film_title", "film_year", "matched_title", "matched_year",
     "confidence", "similarity", "vote_count"]
].to_string(index=False))

                                             film_title  film_year                                           matched_title  matched_year confidence  similarity  vote_count
                                                Monster     2018.0                                                 Monster        2018.0      close       1.000         1.0
                                                  Alpha     2025.0                                                   Alpha        2025.0      close       1.000       140.0
                                                Solaris     2007.0                                                 Solaris        2007.0      close       1.000         1.0
   Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles     1975.0                                                     NaN           NaN   no_match       0.000         NaN
                                  Q: The Winged Serpent     1982.0                                                       Q        1982.0    

In [7]:
lookups = {
    "Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)": "Jeanne Dielman",
    "Precious (2009)":                           "Precious",
    "The Ear (1970)":                            "The Ear",
    "The Night (2021)":                          "The Night",
    "Monster (2018)":                            "Monster",
    "Nausicaä of the Valley of the Wind (1984)": "Nausicaä of the Valley of the Wind",
    "The Murmuring (2022)":                      "The Murmuring",
    "The Civil War on Drugs (2011)":             "The Civil War on Drugs",
    "Sybil (1976)":                              "Sybil",
    "The Castle (1997)":                         "The Castle",
}

current = wl_matches.set_index("film_key")["tmdb_id"]

for key, query in lookups.items():
    year = int(key[-5:-1])
    print(f"=== {key}   current match: {current.get(key, 'held back')}")
    for r in search_film(query, headers=HEADERS):
        release = r.get("release_date") or ""
        y = int(release[:4]) if release[:4].isdigit() else None
        if y is None or abs(y - year) <= 5:
            print(f"  {r['id']:>8}  {y or '????'}  {r.get('vote_count', 0):>6} votes  "
                  f"{r['title']}  /  {r['original_title']}")
    print()

=== Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)   current match: nan
    308191  1975       2 votes  Around Jeanne Dielman  /  Autour de Jeanne Dielman
     44012  1976     410 votes  Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles  /  Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles
   1404470  ????       0 votes  Exercises in Style. Jeanne Dielman  /  Exercises in Style. Jeanne Dielman

=== Precious (2009)   current match: 1772671.0
     25793  2009    1821 votes  Precious: Based on the Novel 'Push' by Sapphire  /  Precious: Based on the Novel 'Push' by Sapphire
   1010417  ????       0 votes  Precious Cargo  /  Precious Cargo

=== The Ear (1970)   current match: 888896.0
    888896  1970       0 votes  The Burning Ear  /  The Burning Ear
    340000  1970       1 votes  The Eye Hears, the Ear Sees  /  The Eye Hears, the Ear Sees
    799285  ????       0 votes  Into the Ear, Everybody  /  Into the Ear, Everybody

=== The Night (2021)   current match: 604360.0


In [8]:
second = [
    ("Ucho",      None),   # The Ear's original Czech title
    ("The Night", 2020),
    ("The Night", 2021),
    ("Monster",   2021),
    ("Monster",   2018),
]

for query, year in second:
    print(f"=== {query!r}, year={year}")
    for r in search_film(query, year, headers=HEADERS)[:8]:
        release = r.get("release_date") or "????"
        print(f"  {r['id']:>8}  {release[:4]}  {r.get('vote_count', 0):>6} votes  "
              f"{r['title']}  /  {r['original_title']}")
    print()

=== 'Ucho', year=None
     88953  1990      74 votes  The Ear  /  Ucho
    334772  1945       8 votes  The Eye & the Ear  /  Oko I Ucho
   1276412  2016       0 votes  The Internal Ear  /  Ucho wewnętrzne
   1064567  1972       0 votes  Kým sa ucho neodbije  /  Kým sa ucho neodbije
     64711  1949     142 votes  Long-Haired Hare  /  Long-Haired Hare
     63899  1977      57 votes  White Bim Black Ear  /  Белый Бим Чёрное ухо
    588336  2016       2 votes  The Mystery of Van Gogh's Ear  /  The Mystery of Van Gogh's Ear
     19316  2004     275 votes  Saving Face  /  Saving Face

=== 'The Night', year=2020
      3112  1955    1905 votes  The Night of the Hunter  /  The Night of the Hunter
    547565  2021    1392 votes  The Night House  /  The Night House
    565743  2019    1239 votes  The Vast of Night  /  The Vast of Night
    526007  2020     932 votes  The Night Clerk  /  The Night Clerk
    640796  2020       3 votes  Into the Night  /  Into the Night
    686245  2020     266 vot

In [9]:
WL_OVERRIDES = {
    "Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles (1975)": 44012,
    "Precious (2009)":  25793,
    "The Ear (1970)":   88953,
    "The Night (2021)": 854531,
    "Monster (2018)":   489932,
}

WL_EXCLUDE = {
    "The New Pope (2020)":          "tv",
    "The Young Pope (2016)":        "tv",
    "Ren Faire (2024)":             "tv",
    "Storm of the Century (1999)":  "tv",
    "Berlin Alexanderplatz (1980)": "tv",
    "Dekalog (1989)":               "tv",
    "Salem's Lot (1979)":           "tv",
    "Little Women (2017)":          "tv",
    "Catch-22 (2019)":              "tv",
    "Sybil (1976)":                 "tv",
    "The Murmuring (2022)":         "not on TMDB",
}

COLLISION_OVERRIDES = {                 # by URI: film_key can't tell these apart
    "https://boxd.it/1sWI": 26891,      # The Castle — Haneke (Das Schloß)
    "https://boxd.it/1Pho": 13852,      # The Castle — Sitch
}

wl_final = wl_matches.merge(films_wl[["film_key", "film_uri"]], on="film_key", how="left")

for key in [*WL_OVERRIDES, *WL_EXCLUDE]:
    if key not in set(wl_final["film_key"]):
        print(f"warning: key not found — {key!r}")

for key, tmdb_id in WL_OVERRIDES.items():
    mask = wl_final["film_key"] == key
    wl_final.loc[mask, "tmdb_id"] = tmdb_id
    wl_final.loc[mask, "confidence"] = "manual_override"

before = len(wl_final)
wl_final = wl_final[~wl_final["film_key"].isin(WL_EXCLUDE)]
n_excluded = before - len(wl_final)

castle = pd.DataFrame({
    "film_key":   collisions["film_key"].values,
    "film_title": collisions["Name"].values,
    "film_year":  collisions["Year"].values,
    "film_uri":   collisions["Letterboxd URI"].values,
})
castle["tmdb_id"]    = castle["film_uri"].map(COLLISION_OVERRIDES)
castle["confidence"] = "manual_override"
assert castle["tmdb_id"].notna().all(), "a collision URI has no override"

wl_final = pd.concat([wl_final, castle], ignore_index=True)

print(f"overrides applied : {len(WL_OVERRIDES)} + {len(castle)} by URI")
print(f"excluded          : {n_excluded}")
print(f"final             : {len(wl_final)} films\n")
print(wl_final["confidence"].value_counts())
print(f"\nmissing tmdb_id   : {wl_final['tmdb_id'].isna().sum()}")
print(f"duplicate tmdb_id : {wl_final['tmdb_id'].duplicated().sum()}")

overrides applied : 5 + 2 by URI
excluded          : 11
final             : 4012 films

confidence
exact              3673
year_off            295
weak                 35
manual_override       7
close                 2
Name: count, dtype: int64

missing tmdb_id   : 0
duplicate tmdb_id : 0


In [10]:
wl_final[wl_final["confidence"] == "year_off"][
    ["film_title", "film_year", "matched_title", "matched_year", "vote_count"]
].sample(10, random_state=0).to_string(index=False)

"                 film_title  film_year               matched_title  matched_year  vote_count\n                 The Bronze     2015.0                  The Bronze        2016.0       404.0\n       The Celluloid Closet     1995.0        The Celluloid Closet        1996.0       120.0\n          Heaven Knows What     2014.0           Heaven Knows What        2015.0       204.0\nAt the First Breath of Wind     2002.0 At the First Breath of Wind        2003.0        15.0\n            In Vanda's Room     2000.0             In Vanda's Room        2001.0        57.0\n             White Material     2009.0              White Material        2010.0       181.0\n                  Ned Rifle     2014.0                   Ned Rifle        2015.0        58.0\n            The Daytrippers     1996.0             The Daytrippers        1997.0       113.0\n           Imagine Me & You     2005.0            Imagine Me & You        2006.0      1136.0\n            35 Shots of Rum     2008.0             35 Shots

In [11]:
WL_DETAILS_CACHE = Path("data/cache/watchlist_details.json")

wl_details = fetch_all_details(wl_final["tmdb_id"], headers=HEADERS, cache_path=WL_DETAILS_CACHE)
print(wl_details.shape)
print(f"missing poster: {wl_details['poster_path'].isna().sum()}")

done — 0 new API calls, 4012 from cache
(4012, 18)
missing poster: 0


In [12]:
print("missing values:")
print(wl_details.isna().sum()[lambda s: s > 0])
print()
print(f"runtime = 0 or null : {((wl_details['runtime'] == 0) | wl_details['runtime'].isna()).sum()}")
print(f"no genres           : {(wl_details['genres'] == '').sum()}")
print(f"no keywords         : {(wl_details['keywords'] == '').sum()}")
print(f"no director         : {wl_details['director'].isna().sum()}")
print(f"no cinematographer  : {wl_details['cinematographer'].isna().sum()}")
print(f"zero vote_count     : {(wl_details['vote_count'] == 0).sum()}")

missing values:
collection_name    3568
cinematographer     100
dtype: int64

runtime = 0 or null : 4
no genres           : 1
no keywords         : 232
no director         : 0
no cinematographer  : 100
zero vote_count     : 20


In [13]:
rated_films = pd.read_csv("data/interim/films_enriched.csv")

cols = ["runtime", "vote_average", "vote_count", "popularity"]
lo, hi = rated_films[cols].min(), rated_films[cols].max()
print(pd.DataFrame({"rated min": lo, "rated max": hi}), "\n")

outside = (wl_details[cols] < lo) | (wl_details[cols] > hi)
print("watchlist films outside the rated range, by column:")
print(outside.sum(), "\n")
print(f"outside on any column: {outside.any(axis=1).sum()}\n")

odd = (wl_details["runtime"].fillna(0) == 0) | (wl_details["genres"] == "") | (wl_details["vote_count"] == 0)
print(wl_details[odd][["tmdb_id", "tmdb_title", "release_date", "runtime", "genres",
                       "vote_count", "vote_average"]].to_string(index=False))

              rated min   rated max
runtime         45.0000    345.0000
vote_average     4.2910      8.6870
vote_count       7.0000  40184.0000
popularity       0.3222    690.8873 

watchlist films outside the rated range, by column:
runtime         65
vote_average    33
vote_count      45
popularity       2
dtype: int64 

outside on any column: 112

 tmdb_id                    tmdb_title release_date  runtime                                  genres  vote_count  vote_average
 1637225             My Brother Jordan   2020-08-19       63                             Documentary           0           0.0
  759517       Benighted but Not Begun   1994-01-01       24                                                   0           0.0
  634154                         Tiger   2011-10-31       70                                   Drama           0           0.0
  209124             How Far Is Heaven   2012-08-23       99                             Documentary           0           0.0
 1259211    

In [14]:
PREDICTION_DATE = pd.Timestamp("2026-09-21")
MIN_VOTES       = 7      # lowest vote count among the rated films
MAX_SHORT       = 40     # Academy definition of a short, in minutes

wl = wl_final.drop(columns=["vote_count"]).merge(wl_details, on="tmdb_id", how="left")
wl["release_date"] = pd.to_datetime(wl["release_date"], errors="coerce")

few_votes = wl["vote_count"] < MIN_VOTES
short     = ~few_votes & (wl["runtime"] <= MAX_SHORT)

upcoming = wl[few_votes
              & (wl["release_date"] > PREDICTION_DATE)
              & (wl["release_date"] <= PREDICTION_DATE + pd.Timedelta(days=90))
              & (wl["runtime"] > 0)].copy()

wl_pred = wl[~few_votes & ~short].copy()

out = (wl_pred[cols] < lo) | (wl_pred[cols] > hi)
wl_pred["out_of_range"] = out.apply(lambda row: "|".join(row.index[row]), axis=1)

print(f"matched films           : {len(wl)}")
print(f"excluded, < {MIN_VOTES} votes     : {few_votes.sum()}")
print(f"excluded, short (<= {MAX_SHORT}) : {short.sum()}")
print(f"to predict              : {len(wl_pred)}")
print(f"  of which flagged      : {(wl_pred['out_of_range'] != '').sum()}")
print(wl_pred.loc[wl_pred["out_of_range"] != "", "out_of_range"].value_counts().to_string())
print(f"\ncoming-soon candidates  : {len(upcoming)}")
print(upcoming[["tmdb_title", "release_date", "runtime"]]
      .sort_values("release_date").to_string(index=False))

matched films           : 4012
excluded, < 7 votes     : 45
excluded, short (<= 40) : 43
to predict              : 3924
  of which flagged      : 24
out_of_range
runtime         11
vote_average    11
popularity       2

coming-soon candidates  : 11
        tmdb_title release_date  runtime
     Possible Love   2026-09-23      165
         Primetime   2026-09-23      110
            Digger   2026-09-30      129
          Ray Gunn   2026-10-10      119
          Minotaur   2026-10-14      135
          Clayface   2026-10-21      108
   Wild Horse Nine   2026-11-04      118
       Paper Tiger   2026-11-12      115
         The Debut   2026-12-10      105
  Dune: Part Three   2026-12-15      140
Avengers: Doomsday   2026-12-16      165


In [15]:
rated_df = pd.read_csv("data/interim/modelling_base.csv", parse_dates=["watched_date"])
rated_df = rated_df.sort_values("watched_date").reset_index(drop=True)
GENRES_ALL = genre_vocabulary(rated_df["genres"])

combined = pd.concat([rated_df, wl_pred.assign(rating=np.nan)], ignore_index=True)
n = len(rated_df)

m_rated, x_rated = genre_history(rated_df, GENRES_ALL)
m_comb,  x_comb  = genre_history(combined, GENRES_ALL)

pd.testing.assert_series_equal(m_comb[:n], m_rated)
pd.testing.assert_series_equal(x_comb[:n], x_rated)
print(f"rated rows identical: {n} viewings")
print(f"genres in vocabulary: {len(GENRES_ALL)}")
print(f"watchlist rows with a genre mean: {m_comb[n:].notna().sum()} of {len(combined) - n}")

rated rows identical: 1192 viewings
genres in vocabulary: 16
watchlist rows with a genre mean: 3828 of 3924


In [16]:
no_genre = combined[n:][m_comb[n:].isna()]
print(no_genre["genres"].fillna("(none)").value_counts().head(10).to_string())
print(f"\nvocabulary: {GENRES_ALL}")

genres
Documentary             68
Western                 26
Documentary|TV Movie     2

vocabulary: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'Thriller', 'War']


In [18]:
LANGUAGES_ALL = top_n_vocabulary(rated_df["original_language"], n=10)
LOG_DEPLOY    = [c for c in LOG_NUMERIC if c != "review_words"]

wl_rows  = wl_pred.assign(rating=np.nan, film_decade=(wl_pred["film_year"] // 10) * 10)
combined = pd.concat([rated_df, wl_rows], ignore_index=True)

F_all   = build_features_F(combined, GENRES_ALL, LANGUAGES_ALL, PLAIN_NUMERIC, LOG_DEPLOY)
H_rated = build_features_H(rated_df, GENRES_ALL, HISTORY_KEYS)
H_comb  = build_features_H(combined, GENRES_ALL, HISTORY_KEYS)

kw_oof, kw_wl = keyword_scores(rated_df, wl_rows)

X_fit = add_keyword_score(pd.concat([F_all[:n], H_rated], axis=1), kw_oof)
X_wl  = add_keyword_score(pd.concat([F_all[n:], H_comb[n:]], axis=1).reset_index(drop=True), kw_wl)

print(f"languages: {LANGUAGES_ALL}")
print(f"X_fit {X_fit.shape}   X_wl {X_wl.shape}")
print(f"columns match: {list(X_fit.columns) == list(X_wl.columns)}")
print(f"nulls — fit {X_fit.isna().sum().sum()}, watchlist {X_wl.isna().sum().sum()}")
print(f"fit rows without a keyword score: {X_fit['kw_score_missing'].sum()}")
print(f"watchlist hist_count_all: {X_wl['hist_count_all'].unique()}")

languages: ['en', 'fr', 'ja', 'ko', 'zh']
X_fit (1192, 48)   X_wl (3924, 48)
columns match: True
nulls — fit 0, watchlist 0
fit rows without a keyword score: 112
watchlist hist_count_all: [1192.]


In [21]:
FINAL_PARAMS = {"n_estimators": 300, "max_features": 0.5, "min_samples_leaf": 5, "random_state": 0}

final_rf = RandomForestRegressor(**FINAL_PARAMS).fit(X_fit, rated_df["rating"])

wl_out = wl_pred.reset_index(drop=True).copy()
wl_out["pred"] = final_rf.predict(X_wl)

genre_cols = [c for c in X_wl.columns if c.startswith("genre_")]
no_genre = X_wl[genre_cols].sum(axis=1) == 0
wl_out.loc[no_genre, "out_of_range"] = (wl_out.loc[no_genre, "out_of_range"]
                                        .replace("", "genre")
                                        .where(lambda s: s == "genre", lambda s: s + "|genre"))

print(wl_out["pred"].describe().round(2).to_string())
print(f"\ncorr with crowd score: {np.corrcoef(wl_out['pred'], wl_out['vote_average'])[0, 1]:.3f}")
print(f"flagged: {(wl_out['out_of_range'] != '').sum()}\n")

show = ["film_title", "film_year", "pred", "vote_average", "out_of_range"]
print("top 15:")
print(wl_out.nlargest(15, "pred")[show].round(2).to_string(index=False))
print("\nbottom 5:")
print(wl_out.nsmallest(5, "pred")[show].round(2).to_string(index=False))

count    3924.00
mean        3.57
std         0.43
min         2.14
25%         3.27
50%         3.60
75%         3.91
max         4.55

corr with crowd score: 0.645
flagged: 119

top 15:
                                 film_title  film_year  pred  vote_average out_of_range
The Human Condition III: A Soldier's Prayer     1961.0  4.55          8.42             
                             Doctor Zhivago     1965.0  4.55          7.60             
                        Fiddler on the Roof     1971.0  4.54          7.74             
   The Human Condition II: Road to Eternity     1959.0  4.53          8.22             
                                  Red Beard     1965.0  4.52          8.20             
             The Good, the Bad and the Ugly     1966.0  4.52          8.47        genre
                        Birdman of Alcatraz     1962.0  4.51          7.50             
               The Bridge on the River Kwai     1957.0  4.51          7.82             
                    

In [22]:
rated_by_dec = rated_df.groupby("film_decade")["rating"].agg(["count", "mean"])
wl_by_dec = (wl_out.assign(film_decade=(wl_out["film_year"] // 10) * 10)
             .groupby("film_decade")["pred"].agg(["count", "mean"]))

by_dec = rated_by_dec.join(wl_by_dec, lsuffix="_rated", rsuffix="_wl", how="outer")
print(by_dec.round(2).to_string())

             count_rated  mean_rated  count_wl  mean_wl
film_decade                                            
1910.0               NaN         NaN         5     3.61
1920.0               2.0        4.00        31     3.99
1930.0               6.0        3.58        62     3.76
1940.0              11.0        4.00        91     3.97
1950.0              24.0        4.15       173     4.00
1960.0              43.0        4.09       283     3.96
1970.0              55.0        4.21       378     3.86
1980.0              94.0        4.05       445     3.77
1990.0             135.0        3.84       586     3.61
2000.0             208.0        3.57       651     3.41
2010.0             277.0        3.53       802     3.35
2020.0             337.0        3.10       417     3.16


In [23]:
yr_lo, yr_hi = rated_df["film_year"].min(), rated_df["film_year"].max()
off_year = (wl_out["film_year"] < yr_lo) | (wl_out["film_year"] > yr_hi)

wl_out.loc[off_year, "out_of_range"] = wl_out.loc[off_year, "out_of_range"].apply(
    lambda s: "film_year" if s == "" else s + "|film_year")

print(f"rated film_year range: {yr_lo:.0f}–{yr_hi:.0f}")
print(f"flagged on film_year: {off_year.sum()}")
print(f"total flagged: {(wl_out['out_of_range'] != '').sum()}")

rated film_year range: 1922–2026
flagged on film_year: 9
total flagged: 128


In [24]:
OUT_COLS = ["film_uri", "film_key", "film_title", "film_year", "tmdb_id",
            "pred", "vote_average", "vote_count", "runtime", "genres", "director",
            "original_language", "release_date", "poster_path", "out_of_range"]

PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

(wl_out[OUT_COLS]
 .sort_values("pred", ascending=False)
 .to_csv(PROCESSED / "watchlist_predictions.csv", index=False))

print(f"saved {len(wl_out)} films -> {PROCESSED / 'watchlist_predictions.csv'}")

saved 3924 films -> data/processed/watchlist_predictions.csv
